# OCR sách Hán bằng TongGuOCR · offline trên Kaggle (4× T4)

Notebook **standalone, Internet Off**. Wheel mã dự án (`vtm_corpus-*.whl`, build từ
nhánh `mduy`) và bộ packages TongGu được cài offline từ Dataset; weights TongGu từ
Dataset `mduy2911/nlp-model` (giống notebook `3_so_sanh_so_chu_ocr`). Interface giống
`1_ocr_paddle`: setup trước, xem danh sách PDF, điền cấu hình rồi **Run All**.

**Settings:** Accelerator **GPU T4 x2/x4**, Internet **Off**. Gắn 3 loại Dataset:
packages TongGu (chứa `paddle_manifest.json`...), models TongGu, PDF sách.

- Bước 1: setup môi trường (tập trung một chỗ): bootstrap `pypdfium2`, cài wheel
  `vtm_corpus`, cài runtime TongGu offline (private Python 3.10, CUDA, mmcv).
- Xem danh sách PDF để điền cấu hình.
- Bước 2: cấu hình (một ô duy nhất, gồm cả cài runtime).
- Bước 3: chạy OCR (checkpoint từng trang, resume khi phiên bị ngắt; 4× T4 sẽ sharding
  trang qua 4 worker).
- Bước 4: xem kết quả trong notebook.


## 1. Setup môi trường

Toàn bộ việc chuẩn bị nằm ở đây, không cần cấu hình trước. Ô đầu cài `pypdfium2`
offline cho kernel và định nghĩa hàm quét Dataset; ô sau cài wheel `vtm_corpus`
(mã dự án, có engine tonggu); ô kế chứa hàm cài runtime TongGu offline từ Dataset
packages (verify manifest -> private Python 3.10 -> wheels -> CUDA toolkit từ bundle
-> build mmcv có ops -> probe `nms_rotated` GPU) và cài lại wheel vào Python riêng
của worker; runtime khỏe được dùng lại.


In [ ]:
# ==== BOOTSTRAP OFFLINE CHO KERNEL ====
# Kernel Kaggle chạy Internet Off: pypdfium2 và wheel mã dự án được cài trực tiếp
# từ wheel nằm trong Dataset; không cần điền cấu hình trước.
import os
import subprocess
import sys
from pathlib import Path

def _walk_input(match, root='/kaggle/input', max_visited=3000, max_hits=50):
    """Quét giới hạn /kaggle/input, trả về file khớp điều kiện (Path)."""
    hits, visited = ([], 0)
    for parent, dirs, files in os.walk(root):
        visited += 1
        if visited > max_visited or len(hits) >= max_hits:
            dirs[:] = []
        hits.extend(str(Path(parent) / f) for f in files if match(f))
    return [Path(h) for h in hits]

def _walk_dirs(root='/kaggle/input', max_visited=3000):
    visited = 0
    for parent, dirs, files in os.walk(root):
        visited += 1
        if visited > max_visited:
            dirs[:] = []
        yield parent, dirs, files

def _ensure_pdfium():
    try:
        import pypdfium2  # noqa: F401
        return
    except ModuleNotFoundError:
        pass
    wheels = _walk_input(lambda f: f.startswith('pypdfium2-') and f.endswith('.whl'))
    if not wheels:
        raise ModuleNotFoundError('Không thấy wheel pypdfium2 trong /kaggle/input; kiểm tra Dataset packages đã gắn.')
    target = '/tmp/vtm_pdfium_site'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', '--no-deps',
                    '--target', target, str(wheels[0])], check=True)
    if target not in sys.path:
        sys.path.insert(0, target)
    print(f'Đã cài pypdfium2 offline: {wheels[0].name} -> {target}')

_ensure_pdfium()


In [ ]:
# ==== CÀI PACKAGE vtm_corpus (offline, từ wheel trong Dataset) ====
# Wheel build từ repo (nhánh mduy, có engine tonggu), upload ở:
#   https://www.kaggle.com/datasets/nthday/tonggu-build
# Notebook tự tìm wheel trong /kaggle/input; cài --no-deps vì dependency nặng
# (torch, mmcv...) đã có trong Dataset packages / private runtime TongGu.
def _find_vtm_wheel():
    hits = _walk_input(lambda f: f.startswith('vtm_corpus-') and f.endswith('.whl'))
    if not hits:
        raise FileNotFoundError('Không thấy wheel vtm_corpus-*.whl trong /kaggle/input; '
                                'gắn Dataset nthday/tonggu-build (hoặc dataset chứa wheel) rồi chạy lại.')
    return hits[0]

wheel = _find_vtm_wheel()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', '--no-deps',
                '--force-reinstall', str(wheel)], check=True)
import vtm_corpus
print('vtm_corpus:', vtm_corpus.__file__)
from vtm_corpus import OCRConfig, ReportConfig, list_pdfs, run_ocr, show_layouts, show_ocr_report, show_pages  # noqa: F401
print('Import OK; engine tonggu sẵn sàng.')


In [ ]:
import argparse
import ast
import base64
import csv
from email.parser import Parser
import gzip
import hashlib
import importlib.metadata as metadata
import importlib.util
import io
import json
import os
from pathlib import Path
import platform
import pprint
import re
import shlex
import shutil
import subprocess
import sys
import tarfile
import tempfile
import zipfile
KERNEL_REPORT_CORRECTIONS = {'1e47c86c9fe706d2e297ea8c0f574716c1d1aadb7366e8de99b195c720c0c746': {'numpy': {'reported': '1.26.4', 'active': '2.1.3'}}}
OPENCV_PACKAGES = ('opencv-python', 'opencv-python-headless', 'opencv-contrib-python', 'opencv-contrib-python-headless')
PRIVATE_SITE_INITIALIZATION = "import sys, site, types\nsys.modules['sitecustomize'] = types.ModuleType('sitecustomize')\nsite.main()\n"
PRIVATE_PYTHON_BOOTSTRAP = PRIVATE_SITE_INITIALIZATION + "\nimport os, runpy\nsys.executable = sys.argv.pop(1)\nargs = sys.argv[1:]\nwhile args and args[0] in ('-u', '-I', '-S', '-E', '-s'):\n    args.pop(0)\nif not args:\n    raise SystemExit('Private Python expects -c, -m, or a script path')\nmode = args.pop(0)\nif mode in ('--version', '-V'):\n    print('Python ' + sys.version.split()[0])\nelif mode == '-c':\n    code = args.pop(0)\n    sys.argv = ['-c', *args]\n    main = types.ModuleType('__main__')\n    main.__dict__.update(__package__=None, __spec__=None)\n    sys.modules['__main__'] = main\n    exec(compile(code, '<private-python>', 'exec'), main.__dict__)\nelif mode == '-m':\n    module = args.pop(0)\n    sys.argv = [module, *args]\n    runpy.run_module(module, run_name='__main__', alter_sys=True)\nelse:\n    sys.argv = [mode, *args]\n    sys.path.insert(0, os.path.dirname(os.path.abspath(mode)))\n    runpy.run_path(mode, run_name='__main__')\n"

def isolated_python_command(python, args):
    return [python, '-u', '-I', '-S', '-c', PRIVATE_PYTHON_BOOTSTRAP, python, *args]

def host_pip_command(args):
    return isolated_python_command(sys.executable, ['-m', 'pip', *args])

def command_description(args):
    return ' '.join(('<private Python bootstrap>' if str(arg) == PRIVATE_PYTHON_BOOTSTRAP else str(arg) for arg in args))

def sha256(path):
    hasher = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            hasher.update(chunk)
    return hasher.hexdigest()

def checked_path(root, relative):
    path = (root / relative).resolve()
    if not path.is_relative_to(root.resolve()):
        raise ValueError(f'Path outside bundle: {relative}')
    return path

def source_tree_digest(files):
    data = json.dumps(sorted(files, key=lambda r: r['path']), sort_keys=True, separators=(',', ':')).encode()
    return hashlib.sha256(data).hexdigest()

class SourceTreeError(ValueError):

    def __init__(self, message, report):
        self.report = report
        missing = report.get('unresolved_nonempty', [])
        changed = report.get('changed', [])
        details = [f"{report['source']}: expected={report['expected_files']}, actual={report['actual_files']}"]
        if missing:
            details.append(f'Missing nonempty files ({len(missing)}): ' + ', '.join(missing[:12]))
        if changed:
            details.append(f'Integrity failure ({len(changed)}): ' + ', '.join((r['path'] for r in changed[:12])))
        super().__init__(message + '\n' + '\n'.join(details))

def resolve_source_tree(bundle, entry):
    """Resolve only content verified by the manifest; never write to Kaggle Input."""
    directory = checked_path(bundle, entry['path'])
    files = entry['files']
    expected = {r['path'] for r in files}
    if len(files) != len(expected) or source_tree_digest(files) != entry['sha256']:
        raise ValueError(f"Invalid source tree manifest: {entry['path']}")
    actual = {p.relative_to(directory).as_posix(): p for p in directory.rglob('*') if p.is_file()}
    folded = {}
    for name in actual:
        folded.setdefault(name.casefold(), []).append(name)
    report = {'source': entry['path'], 'directory': str(directory), 'expected_files': len(expected), 'actual_files': len(actual), 'expected_bytes': sum((r['bytes'] for r in files)), 'declared_bytes': entry['bytes'], 'missing_uploaded': sorted(expected - actual.keys()), 'ignored_extras': sorted(actual.keys() - expected), 'recreated_empty': [], 'resolved_names': [], 'reused_payloads': [], 'omitted_bytecode': [], 'unresolved_nonempty': [], 'changed': []}
    if report['expected_bytes'] != entry['bytes']:
        raise SourceTreeError('Invalid source tree byte count', report)
    resolved, verified, cache, pending = ({}, {}, {}, [])

    def matches(name, record):
        path = checked_path(directory, name)
        if path.stat().st_size != record['bytes']:
            return False
        if path not in cache:
            cache[path] = sha256(path)
        return cache[path] == record['sha256']
    for record in files:
        name = record['path']
        checked_path(directory, name)
        if name in actual:
            if not matches(name, record):
                report['changed'].append({'path': name, 'expected_bytes': record['bytes'], 'actual_bytes': actual[name].stat().st_size})
                continue
            chosen = name
        else:
            candidates = list(folded.get(name.casefold(), []))
            if record.get('restore_path') and (not record.get('restore_as_gzip')):
                candidates.extend(folded.get(record['restore_path'].casefold(), []))
            chosen = next((candidate for candidate in candidates if matches(candidate, record)), None)
            if chosen is None:
                if record['bytes'] == 0 and record['sha256'] == hashlib.sha256(b'').hexdigest():
                    resolved[name] = None
                    report['recreated_empty'].append(name)
                else:
                    pending.append(record)
                continue
            report['resolved_names'].append({'expected': name, 'actual': chosen})
        resolved[name] = checked_path(directory, chosen)
        verified[record['bytes'], record['sha256']] = resolved[name]
    unavailable = []
    for record in pending:
        donor = verified.get((record['bytes'], record['sha256']))
        if donor is None:
            unavailable.append(record)
        else:
            resolved[record['path']] = donor
            report['reused_payloads'].append({'expected': record['path'], 'actual': donor.relative_to(directory).as_posix()})
    logical_records = {r.get('restore_path', r['path']): r for r in files}
    for record in unavailable:
        source_record = None
        if entry.get('package') == 'python-runtime' and Path(record['path']).suffix == '.pyc' and (not record.get('restore_as_gzip')):
            try:
                source_name = Path(importlib.util.source_from_cache(record.get('restore_path', record['path']))).as_posix()
            except ValueError:
                pass
            else:
                source_record = logical_records.get(source_name)
        if source_record is not None and source_record['path'] in resolved and (not source_record.get('restore_as_gzip')):
            report['omitted_bytecode'].append({'path': record['path'], 'source': source_record['path']})
        else:
            report['unresolved_nonempty'].append(record['path'])
    if report['unresolved_nonempty'] or report['changed']:
        raise SourceTreeError('Incomplete or changed source tree', report)
    return (resolved, report)

def verify_source_tree(bundle, entry):
    _, report = resolve_source_tree(bundle, entry)
    for key, label in (('recreated_empty', 'file rỗng sẽ tạo lại trong /tmp'), ('resolved_names', 'tên file được đối chiếu bằng SHA-256'), ('reused_payloads', 'payload trùng checksum được dùng lại'), ('omitted_bytecode', 'cache .pyc thiếu; dùng file .py đã xác minh'), ('ignored_extras', 'file thừa không được chép vào runtime')):
        if report[key]:
            print(f"{entry['path']}: {len(report[key])} {label}.", flush=True)
    return report

def copy_source_tree(bundle, entry, target):
    """Copy plain uploaded files into writable storage and restore Linux modes."""
    resolved, report = resolve_source_tree(bundle, entry)
    omitted = {r['path'] for r in report['omitted_bytecode']}
    target = Path(target)
    target.mkdir(parents=True, exist_ok=True)
    for record in entry['files']:
        if record['path'] in omitted:
            continue
        source = resolved[record['path']]
        destination = checked_path(target, record.get('restore_path', record['path']))
        destination.parent.mkdir(parents=True, exist_ok=True)
        if source is None:
            if record.get('restore_as_gzip'):
                with gzip.open(destination, 'wb'):
                    pass
            else:
                destination.write_bytes(b'')
        elif record.get('restore_as_gzip'):
            with source.open('rb') as stream, gzip.open(destination, 'wb') as output:
                shutil.copyfileobj(stream, output)
        else:
            shutil.copyfile(source, destination)
        destination.chmod(record['mode'])
    return target

def prepare_python_runtime(bundle, entry, directory):
    if entry.get('layout') == 'expanded_directory':
        copy_source_tree(bundle, entry, directory / 'python')
    else:
        extract_tar(checked_path(bundle, entry['path']), directory)

def prepare_build_source(bundle, entry, runtime):
    if entry.get('layout') == 'expanded_directory':
        return copy_source_tree(bundle, entry, runtime / 'build_sources' / Path(entry['path']).name)
    return checked_path(bundle, entry['path'])

def prepare_tonggu_checkout(bundle, entry, runtime):
    source_root = runtime / 'tonggu_source'
    if entry.get('layout') == 'expanded_directory':
        return copy_source_tree(bundle, entry, source_root / 'TongGuOCR')
    source_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(checked_path(bundle, entry['path'])) as archive:
        for info in archive.infolist():
            checked_path(source_root, info.filename)
        archive.extractall(source_root)
    return next(source_root.glob('TongGuOCR-*/tongguocr/pipeline.py')).parents[1]

def verify(bundle, *, report_path=None):
    records = {}
    reports = []
    groups = ['paddle', 'tonggu', 'system_assets']
    if (bundle / 'app_manifest.json').is_file():
        groups.append('app')
    for name in groups:
        path = bundle / f'{name}_manifest.json'
        if not path.is_file():
            raise FileNotFoundError(f'Incomplete bundle: {path}')
        manifest = json.loads(path.read_text(encoding='utf-8'))
        if manifest.get('failed'):
            raise ValueError(f'Bundle has failed downloads: {name}')
        if name == 'tonggu' and any((r['package'].lower().replace('_', '-') == 'nvidia-nccl-cu13' for r in manifest.get('reused', []))):
            raise ValueError('Dataset packages còn dùng lại NCCL từ kernel. Cập nhật Dataset packages bằng bộ đã bổ sung wheel NCCL riêng cho TongGu, rồi gắn phiên bản mới vào notebook. Chỉ thay ô setup chưa đủ cho lỗi checksum NCCL.')
        for entry in manifest['records']:
            if entry.get('layout') == 'expanded_directory':
                try:
                    reports.append(verify_source_tree(bundle, entry))
                except SourceTreeError as exc:
                    if report_path is not None:
                        report_path = Path(report_path)
                        report_path.parent.mkdir(parents=True, exist_ok=True)
                        report_path.write_text(json.dumps({'ok': False, 'source_trees': reports + [exc.report]}, ensure_ascii=False, indent=2), encoding='utf-8')
                    raise
                records[entry['path']] = entry
                continue
            path = checked_path(bundle, entry['path'])
            if path.stat().st_size != entry['bytes'] or sha256(path) != entry['sha256']:
                raise ValueError(f'Integrity failure: {path.name}')
            records[entry['path']] = entry
    if report_path is not None:
        report_path = Path(report_path)
        report_path.parent.mkdir(parents=True, exist_ok=True)
        report_path.write_text(json.dumps({'ok': True, 'source_trees': reports}, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f"Verified {len(records)} artifacts, {sum((r['bytes'] for r in records.values())) / 1024 ** 3:.2f} GiB", flush=True)
    return list(records.values())

def extract_tar(path, target):
    target.mkdir(parents=True, exist_ok=True)
    with tarfile.open(path) as archive:
        archive.extractall(target, filter='data')

def target_site(python, runtime):
    code = PRIVATE_SITE_INITIALIZATION + "\nimport json,sysconfig; print(json.dumps([sysconfig.get_paths()['purelib'],sys.version.split()[0]]))"
    site, version = json.loads(subprocess.check_output([str(python), '-u', '-I', '-S', '-c', code], env=native_environment(), text=True).strip())
    site = Path(site).resolve()
    if not site.is_relative_to(runtime.resolve()):
        raise ValueError(f'Target interpreter is outside the private runtime: {site}')
    site.mkdir(parents=True, exist_ok=True)
    return (site, version)

def clear_kernel_links(site):
    ledger = site / '.vtm_kernel_links.json'
    if not ledger.is_file():
        return
    for relative in json.loads(ledger.read_text(encoding='utf-8')):
        path = site / relative
        if not path.parent.resolve().is_relative_to(site.resolve()):
            raise ValueError(f'Kernel link escapes target site: {relative}')
        if path.is_symlink():
            path.unlink()
    ledger.unlink()

def clear_reused_packages(site, records):
    """Remove previous private copies, including a failed install; never follow kernel links."""
    from packaging.utils import canonicalize_name
    names = {canonicalize_name(r['package']) for r in records}
    owned, protected = (set(), set())
    for dist in metadata.distributions(path=[str(site)]):
        name = canonicalize_name(dist.metadata.get('Name', ''))
        paths = owned if name in names else protected
        for row in csv.reader(io.StringIO(dist.read_text('RECORD'))):
            if row:
                paths.add(row[0])
    ledger = site / '.vtm_kernel_files.json'
    if ledger.is_file():
        owned.update(json.loads(ledger.read_text(encoding='utf-8')))
    targets = []
    for relative in owned - protected:
        path = Path(relative)
        if path.is_absolute() or '..' in path.parts:
            continue
        target = site / path
        if not target.parent.resolve().is_relative_to(site.resolve()):
            raise ValueError(f'Reused package cleanup escapes private site: {relative}')
        if target.is_symlink() or target.is_file():
            targets.append(target)
    for target in targets:
        target.unlink()
    if ledger.is_file():
        ledger.unlink()
    clear_kernel_links(site)

def private_kernel_cleanup_records(manifest):
    """Clean kernel copies; let the dedicated cv2 repair own its shared namespace."""
    from packaging.utils import canonicalize_name
    records = [*manifest.get('reused', []), *manifest.get('kernel_overlap_repairs', [])]
    shipped = {canonicalize_name(r['package']) for r in manifest.get('records', [])}
    if shipped.intersection(OPENCV_PACKAGES):
        records = [r for r in records if canonicalize_name(r['package']) not in OPENCV_PACKAGES]
    return list({canonicalize_name(r['package']): r for r in records}.values())

def opencv_metadata_directories(site):
    """Find even damaged OpenCV metadata without relying on importer caches."""
    from packaging.utils import canonicalize_name
    return [path for path in Path(site).glob('*.dist-info') if canonicalize_name(path.name[:-len('.dist-info')].rsplit('-', 1)[0]) in OPENCV_PACKAGES]

def clear_private_opencv_files(site, runtime):
    """Remove only OpenCV-owned namespaces in a verified private site."""
    site, runtime = (Path(site).resolve(), Path(runtime).resolve())
    if site == runtime or not site.is_relative_to(runtime):
        raise ValueError(f'OpenCV cleanup site is outside private runtime: {site}')
    targets = [site / 'cv2', site / 'cv2.py', *site.glob('cv2.*.so'), *site.glob('cv2*.pyd'), *opencv_metadata_directories(site)]
    targets.extend((site / (name.replace('-', '_') + '.libs') for name in OPENCV_PACKAGES))
    for target in targets:
        if target.parent.resolve() != site:
            raise ValueError(f'OpenCV cleanup target escapes private site: {target}')
        if target.is_symlink():
            target.unlink()
        elif target.exists():
            resolved = target.resolve()
            if resolved == site or not resolved.is_relative_to(site):
                raise ValueError(f'OpenCV cleanup target escapes private site: {target}')
            if target.is_dir():
                shutil.rmtree(target)
            else:
                target.unlink()

def audit_private_opencv(wheel, site):
    """Compare the installed cv2 payload and vendor libraries against wheel RECORD."""
    site = Path(site).resolve()
    problems, checked = ([], 0)
    with zipfile.ZipFile(wheel) as archive:
        record_name = next((n for n in archive.namelist() if n.endswith('.dist-info/RECORD') and n.count('/') == 1))
        expected_info = record_name.rsplit('/', 1)[0]
        expected_files = set(archive.namelist())
        installed_record = site / record_name
        if installed_record.is_symlink() or not installed_record.is_file():
            problems.append({'path': record_name, 'reason': 'missing_record'})
        for row in csv.reader(io.StringIO(archive.read(record_name).decode('utf-8'))):
            if not row or row[0] == record_name:
                continue
            name, checksum, size = row
            relative = Path(name)
            if relative.is_absolute() or '..' in relative.parts or (not checksum):
                raise ValueError(f'Unsafe or unverifiable OpenCV wheel entry: {name}')
            target = site / relative
            if not target.parent.resolve().is_relative_to(site):
                problems.append({'path': name, 'reason': 'parent_symlink'})
                continue
            if target.is_symlink() or not target.is_file():
                problems.append({'path': name, 'reason': 'linked_or_missing'})
                continue
            algorithm, expected_hash = checksum.split('=', 1)
            if size and target.stat().st_size != int(size):
                problems.append({'path': name, 'reason': 'size'})
                continue
            with target.open('rb') as stream:
                actual = base64.urlsafe_b64encode(hashlib.file_digest(stream, algorithm).digest()).rstrip(b'=').decode()
            if actual != expected_hash:
                problems.append({'path': name, 'reason': 'checksum'})
            checked += 1
        for metadata_dir in opencv_metadata_directories(site):
            if metadata_dir.name != expected_info and (metadata_dir.is_file() or any(metadata_dir.iterdir())):
                problems.append({'path': metadata_dir.name, 'reason': 'other_provider'})
        cv2_root = site / 'cv2'
        if cv2_root.is_dir() and cv2_root.resolve().is_relative_to(site):
            for path in cv2_root.rglob('*'):
                name = path.relative_to(site).as_posix()
                if path.is_file() and path.suffix in {'.py', '.so', '.pyd'} and (name not in expected_files) and ('__pycache__' not in path.parts):
                    problems.append({'path': name, 'reason': 'untracked_loader_or_binary'})
        for path in [site / 'cv2.py', *site.glob('cv2.*.so'), *site.glob('cv2*.pyd')]:
            if path.is_file() and path.name not in expected_files:
                problems.append({'path': path.name, 'reason': 'legacy_root_module'})
    return {'wheel': str(wheel), 'site': str(site), 'checked_files': checked, 'problems': problems}
PRIVATE_OPENCV_PROBE = "import importlib.metadata as md, json, sys\nfrom pathlib import Path\nsite, expected, version, output = sys.argv[1:5]\nreport = {'package': expected, 'version': version}\ntry:\n    import cv2, numpy as np\n    report.update(origin=getattr(cv2,'__file__',None), cv2_version=getattr(cv2,'__version__',None))\n    needed = ['resize','cvtColor','warpAffine','getRotationMatrix2D','findContours','threshold',\n              'minAreaRect','boxPoints','copyMakeBorder','imread','imwrite','imencode','imdecode']\n    missing = [name for name in needed if not callable(getattr(cv2,name,None))]\n    report['missing_api'] = missing\n    assert not missing, 'Incomplete cv2 API: ' + ', '.join(missing)\n    assert report['origin'] and Path(report['origin']).resolve().is_relative_to(Path(site).resolve()), report\n    variants = {'opencv-python','opencv-python-headless','opencv-contrib-python','opencv-contrib-python-headless'}\n    providers = {d.metadata.get('Name','').lower().replace('_','-'): d.version\n                 for d in md.distributions(path=[site]) if d.metadata.get('Name','').lower().replace('_','-') in variants}\n    report['providers'] = providers\n    assert providers == {expected:version}, providers\n    image = np.zeros((8,8,3),dtype=np.uint8); image[2:6,2:6] = 255\n    assert cv2.resize(image,(16,16)).shape == (16,16,3)\n    gray = cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)\n    _, binary = cv2.threshold(gray,0,255,cv2.THRESH_BINARY | cv2.THRESH_OTSU)\n    contours, _ = cv2.findContours(binary,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)\n    assert contours and cv2.boxPoints(cv2.minAreaRect(contours[0])).shape == (4,2)\n    matrix = cv2.getRotationMatrix2D((4.,4.),0.,1.)\n    assert cv2.warpAffine(image,matrix,(8,8)).shape == image.shape\n    assert cv2.copyMakeBorder(image,1,1,1,1,cv2.BORDER_CONSTANT).shape == (10,10,3)\n    ok, encoded = cv2.imencode('.png',image)\n    assert ok and cv2.imdecode(encoded,cv2.IMREAD_COLOR).shape == image.shape\n    report['ok'] = True\n    print(json.dumps(report))\nexcept Exception as exc:\n    report.update(ok=False,error=repr(exc),sys_path=sys.path)\n    raise\nfinally:\n    Path(output).write_text(json.dumps(report,indent=2),encoding='utf-8')\n"

def ensure_private_opencv(bundle, manifest, site, python, environment, runtime):
    """Repair partial OpenCV installs even when pip considers their metadata satisfied."""
    from packaging.utils import canonicalize_name
    records = [r for r in manifest.get('records', []) if canonicalize_name(r['package']) in OPENCV_PACKAGES]
    if not records:
        return None
    if len(records) != 1:
        raise ValueError('A private runtime must ship exactly one OpenCV provider')
    record = records[0]
    wheel = checked_path(Path(bundle), record['path'])
    if sha256(wheel) != record['sha256']:
        raise ValueError(f'OpenCV repair wheel checksum differs: {wheel}')
    site, runtime = (Path(site).resolve(), Path(runtime).resolve())
    if site == runtime or not site.is_relative_to(runtime):
        raise ValueError(f'OpenCV site is outside private runtime: {site}')
    audit = audit_private_opencv(wheel, site)
    repaired = bool(audit['problems'])
    if repaired:
        print(f"Repair incomplete OpenCV payload: {len(audit['problems'])} mismatches; first: {audit['problems'][:5]}", flush=True)
        clear_private_opencv_files(site, runtime)
        run(host_pip_command(['--python', python, 'install', '--no-index', '--no-deps', '--force-reinstall', wheel]), environment)
        audit = audit_private_opencv(wheel, site)
    audit['repaired'] = repaired
    audit_path = site / '.vtm_opencv_payload.json'
    audit_path.write_text(json.dumps(audit, indent=2), encoding='utf-8')
    if audit['problems']:
        raise RuntimeError(f'OpenCV payload remains incomplete after reinstall: {audit_path}')
    probe = site / '.vtm_opencv_probe.json'
    run([python, '-c', PRIVATE_OPENCV_PROBE, site, canonicalize_name(record['package']), record['version'], probe], environment)
    print(f"OpenCV payload verified: {audit['checked_files']} files; probe: {probe}", flush=True)
    return str(probe)

def kernel_distribution_compatible(dist, name, target_version):
    from packaging.specifiers import SpecifierSet
    from packaging.tags import compatible_tags, cpython_tags, parse_tag, platform_tags
    requires_python = dist.metadata.get('Requires-Python')
    if requires_python and (not SpecifierSet(requires_python).contains(target_version)):
        raise ValueError(f'Kernel package requires another Python: {name} {requires_python}, target {target_version}')
    minor = int(target_version.split('.')[1])
    platforms = list(platform_tags())
    tags = set(cpython_tags((3, minor), abis=[f'cp3{minor}'], platforms=platforms))
    tags.update(compatible_tags((3, minor), interpreter=f'cp3{minor}', platforms=platforms))
    wheel = dist.read_text('WHEEL')
    wheel_tags = set().union(*(parse_tag(line[5:].strip()) for line in wheel.splitlines() if line.startswith('Tag: ')))
    if (wheel_tags or minor != sys.version_info.minor) and (not wheel_tags & tags):
        raise ValueError(f'Kernel package ABI cannot be reused in Python {target_version}: {name}')

def plan_kernel_reuse(bundle, manifest, target_version, report_path):
    """Check live metadata and its dependency closure against the fixed offline artifacts."""
    from packaging.markers import default_environment
    from packaging.requirements import Requirement
    from packaging.utils import canonicalize_name
    from packaging.version import Version
    environment = {**default_environment(), 'python_version': '.'.join(target_version.split('.')[:2]), 'python_full_version': target_version, 'os_name': 'posix', 'sys_platform': 'linux', 'platform_system': 'Linux', 'platform_machine': 'x86_64', 'implementation_name': 'cpython'}
    selected, live, original = ({}, {}, {canonicalize_name(r['package']): r for r in manifest.get('reused', [])})
    extras = {canonicalize_name(n): set(values) for n, values in manifest.get('extras', {}).items()}
    report = {'target_python': target_version, 'version_changes': [], 'added_dependencies': [], 'excluded_packages': [], 'errors': []}
    protected = {'torch', 'torchvision', 'transformers', 'numpy', 'paddlepaddle-gpu', 'paddleocr', 'paddlex'}
    corrections = KERNEL_REPORT_CORRECTIONS.get(manifest.get('base_report_sha256'), {})

    def select_kernel(name):
        if name in OPENCV_PACKAGES:
            existing = set(selected).intersection(OPENCV_PACKAGES) - {name}
            if existing:
                raise ValueError(f'Multiple OpenCV providers for cv2: {name} conflicts with {sorted(existing)}')
        try:
            dist = metadata.distribution(name)
        except metadata.PackageNotFoundError as exc:
            raise ValueError(f'Offline dependency unavailable in kernel: {name}') from exc
        if canonicalize_name(dist.metadata.get('Name', '')) != name:
            raise ValueError(f'Kernel distribution identity differs: {name}')
        kernel_distribution_compatible(dist, name, target_version)
        prior = original.get(name)
        baseline = prior['version'] if prior else None
        correction = corrections.get(name, {})
        if baseline == correction.get('reported'):
            baseline = correction.get('active', baseline)
        if prior and (name in protected or name.startswith('nvidia-')) and (Version(dist.version) != Version(baseline)):
            raise ValueError(f'Core kernel package changed: {name} expected {baseline}, got {dist.version}')
        if prior and Version(dist.version) != Version(prior['version']):
            report['version_changes'].append({'package': name, 'reported': prior['version'], 'actual': dist.version})
        if prior is None:
            report['added_dependencies'].append({'package': name, 'version': dist.version})
        live[name] = {'package': name, 'version': dist.version, 'reported_version': prior['version'] if prior else None}
        selected[name] = {'version': dist.version, 'requires': dist.metadata.get_all('Requires-Dist', [])}
    try:
        for record in manifest['records']:
            name = canonicalize_name(record['package'])
            requires = record.get('requires', [])
            if record['kind'] == 'wheel':
                with zipfile.ZipFile(checked_path(bundle, record['path'])) as archive:
                    candidates = [n for n in archive.namelist() if n.endswith('.dist-info/METADATA') and n.count('/') == 1]
                    if len(candidates) != 1:
                        raise ValueError(f"Invalid wheel metadata: {record['path']}")
                    data = Parser().parsestr(archive.read(candidates[0]).decode('utf-8'))
                if canonicalize_name(data['Name']) != name or Version(data['Version']) != Version(record['version']):
                    raise ValueError(f"Offline wheel identity differs: {record['path']}")
                requires = data.get_all('Requires-Dist', [])
            selected[name] = {'version': record['version'], 'requires': requires}
        shipped_opencv = set(selected).intersection(OPENCV_PACKAGES)
        if len(shipped_opencv) > 1:
            raise ValueError(f'Multiple shipped OpenCV providers for cv2: {sorted(shipped_opencv)}')
        for name in original:
            if name in selected:
                raise ValueError(f'Both shipped and reused package: {name}')
            if shipped_opencv and name in OPENCV_PACKAGES:
                report['excluded_packages'].append({'package': name, 'reason': f'cv2 is supplied by the shipped {next(iter(shipped_opencv))} wheel'})
                continue
            select_kernel(name)
        requirements = [('root', Requirement(text)) for text in manifest.get('roots', [])]
        changed = True
        while changed:
            changed = False
            for name, record in list(selected.items()):
                for text in record['requires']:
                    req = Requirement(text)
                    if req.marker and (not any((req.marker.evaluate({**environment, 'extra': extra}) for extra in {'', *extras.get(name, set())}))):
                        continue
                    if (name, req) not in requirements:
                        requirements.append((name, req))
                        changed = True
            for parent, req in requirements:
                if parent == 'root' and req.marker and (not req.marker.evaluate({**environment, 'extra': ''})):
                    continue
                if req.url:
                    raise ValueError(f'Offline URL dependency is unsupported: {parent} -> {req}')
                name = canonicalize_name(req.name)
                if name not in selected:
                    try:
                        select_kernel(name)
                    except ValueError as exc:
                        raise ValueError(f'{parent} -> {req}: {exc}') from exc
                    changed = True
                if not req.specifier.contains(selected[name]['version'], prereleases=True):
                    raise ValueError(f"Unsatisfied offline dependency: {parent} -> {req}; have {selected[name]['version']}")
                active = extras.setdefault(name, set())
                if not req.extras <= active:
                    active.update(req.extras)
                    changed = True
        report.update(ok=True, reused=list(live.values()), dependency_edges_checked=len(requirements), opencv_provider=next(iter(set(selected).intersection(OPENCV_PACKAGES)), None))
    except Exception as exc:
        report.update(ok=False)
        report['errors'].append(str(exc))
        raise
    finally:
        report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    for change in report['version_changes']:
        print(f"Compatible kernel update: {change['package']} {change['reported']} -> {change['actual']}", flush=True)
    if report['opencv_provider']:
        print(f"OpenCV provider: {report['opencv_provider']}; excluded {len(report['excluded_packages'])} kernel variants", flush=True)
    print(f'Kernel dependency plan: {len(live)} packages, {len(requirements)} dependency constraints checked', flush=True)
    return list(live.values())

def reuse_kernel_packages(records, site, target_version, *, bundle=None, manifest=None):
    """Copy Python files and link ABI-compatible native files, never add cp313 site to cp310."""
    from packaging.version import Version
    if manifest is not None:
        records = plan_kernel_reuse(bundle, manifest, target_version, site / '.vtm_kernel_plan.json')
    links, copied, reused = ([], [], [])
    try:
        for expected in records:
            dist = metadata.distribution(expected['package'])
            if Version(dist.version) != Version(expected['version']):
                raise ValueError(f"Kernel package changed: {expected['package']} expected {expected['version']}, got {dist.version}")
            kernel_distribution_compatible(dist, expected['package'], target_version)
            source_root = Path(dist.locate_file('')).resolve()
            record_text = dist.read_text('RECORD')
            if not record_text:
                raise ValueError(f"Kernel package has no verifiable RECORD: {expected['package']}")
            count = 0
            for row in csv.reader(io.StringIO(record_text)):
                if not row:
                    continue
                item, checksum, _size = row
                relative = Path(item)
                if relative.is_absolute() or '..' in relative.parts or '__pycache__' in relative.parts or (relative.suffix == '.pyc'):
                    continue
                source = Path(dist.locate_file(item)).resolve()
                if not source.is_relative_to(source_root) or not source.is_file():
                    raise ValueError(f'Kernel package file unavailable: {item}')
                if checksum:
                    algorithm, expected_hash = checksum.split('=', 1)
                    hasher = hashlib.new(algorithm)
                    with source.open('rb') as stream:
                        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
                            hasher.update(block)
                    actual = base64.urlsafe_b64encode(hasher.digest()).rstrip(b'=').decode()
                    if actual != expected_hash:
                        raise ValueError(f"Kernel RECORD checksum differs: {expected['package']}/{item}")
                elif '.so' in source.name:
                    raise ValueError(f'Unverifiable native kernel file: {item}')
                target = site / relative
                if not target.parent.resolve().is_relative_to(site.resolve()):
                    raise ValueError(f'Reused package file escapes private site: {relative}')
                target.parent.mkdir(parents=True, exist_ok=True)
                if target.exists():
                    if sha256(target) != sha256(source):
                        raise ValueError(f'Package files overlap with different bytes: {relative}')
                    continue
                if '.so' in source.name:
                    target.symlink_to(source)
                    links.append(relative.as_posix())
                else:
                    shutil.copy2(source, target)
                copied.append(relative.as_posix())
                count += 1
            reused.append({'package': expected['package'], 'version': dist.version, 'reported_version': expected.get('reported_version', expected['version']), 'files': count})
    finally:
        (site / '.vtm_kernel_links.json').write_text(json.dumps(links, indent=2), encoding='utf-8')
        (site / '.vtm_kernel_files.json').write_text(json.dumps(copied, indent=2), encoding='utf-8')
    print(f'Reused {len(reused)} kernel distributions in Python {target_version}', flush=True)
    return reused

def component_manifest(bundle, name):
    path = bundle / f'{name}_manifest.json'
    return json.loads(path.read_text(encoding='utf-8')) if path.is_file() else {}

def private_python(python, site):
    """Use the same private CUDA library search order for probes and OCR workers."""
    libraries = sorted({str(path.parent) for path in (site / 'nvidia').rglob('*.so*') if path.is_file()})
    wrapper = python.with_name('vtm-python')
    prefix = shlex.quote(':'.join(libraries))
    script = '#!/bin/sh\nunset PYTHONPATH PYTHONHOME PYTHONUSERBASE\nexport PYTHONNOUSERSITE=1\nexport MPLBACKEND=Agg\n'
    if libraries:
        script += 'export LD_LIBRARY_PATH=' + prefix + '"${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}"\n'
    command = isolated_python_command(python, [])
    command[-1] = wrapper
    script += 'exec ' + ' '.join((shlex.quote(str(arg)) for arg in command)) + ' "$@"\n'
    wrapper.write_text(script, encoding='utf-8', newline='\n')
    wrapper.chmod(493)
    return wrapper

def native_environment(cuda=None):
    env = os.environ.copy()
    for name in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONUSERBASE', '_PYTHON_SYSCONFIGDATA_NAME', '_PYTHON_HOST_PLATFORM'):
        env.pop(name, None)
    env.update(PIP_NO_INDEX='1', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', HF_HUB_DISABLE_TELEMETRY='1', PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK='True', PYTHONNOUSERSITE='1', MPLBACKEND='Agg')
    if cuda:
        env.update(CUDA_HOME=str(cuda), CUDA_PATH=str(cuda), MAX_JOBS=os.environ.get('MAX_JOBS', '2'))
        env['PATH'] = str(cuda / 'bin') + os.pathsep + env.get('PATH', '')
        env['LD_LIBRARY_PATH'] = str(cuda / 'lib64') + os.pathsep + env.get('LD_LIBRARY_PATH', '')
    return env

def configure_cuda_architecture(python, environment):
    """Compile native ops for the GPUs in this session, including L4 SM 8.9."""
    source = "import json,torch\nassert torch.cuda.is_available(), 'Native CUDA build requires a visible GPU'\nprint(json.dumps(sorted({f'{major}.{minor}' for major,minor in\n    [torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())]})))\n"
    output = subprocess.check_output([str(python), '-c', source], env=environment, text=True)
    arches = json.loads(output.splitlines()[-1])
    if not arches or any((not re.fullmatch('\\d+\\.\\d+', arch) for arch in arches)):
        raise RuntimeError('Cannot determine CUDA architecture for native build')
    environment['TORCH_CUDA_ARCH_LIST'] = ';'.join(arches)
    return environment['TORCH_CUDA_ARCH_LIST']

def repair_python_build_config(python_root, environment):
    """Relocate standalone Python's build paths in the writable runtime only."""
    python_root = Path(python_root).resolve()
    configs = list(python_root.glob('lib/python*/_sysconfigdata*.py'))
    if len(configs) != 1 or not configs[0].resolve().is_relative_to(python_root):
        raise ValueError(f'Cannot identify private Python build config: {python_root}')
    path = configs[0]
    tree = ast.parse(path.read_text(encoding='utf-8'))
    values = next((ast.literal_eval(n.value) for n in tree.body if isinstance(n, ast.Assign) and any((isinstance(t, ast.Name) and t.id == 'build_time_vars' for t in n.targets))))
    previous = dict(values)
    old_prefix = values.get('prefix', '/install')
    if old_prefix != str(python_root):
        pattern = re.compile(re.escape(old_prefix) + '(?=/|$|[\\s:])')
        values = {key: pattern.sub(lambda _: str(python_root), value) if isinstance(value, str) else value for key, value in values.items()}
    values.update(CC=environment['CC'] + ' -pthread', CXX=environment['CXX'] + ' -pthread', LDSHARED=environment['LDSHARED'], LDCXXSHARED=environment['LDCXXSHARED'], BLDSHARED=environment['LDSHARED'])
    changed = sorted((key for key in values if values[key] != previous.get(key)))
    if changed:
        path.write_text('build_time_vars = ' + pprint.pformat(values, sort_dicts=True, width=110) + '\n', encoding='utf-8')
        for cached in (path.parent / '__pycache__').glob(path.stem + '.*.pyc'):
            if cached.is_file():
                cached.unlink()
    include = Path(values['INCLUDEPY'])
    if not include.is_relative_to(python_root) or not (include / 'Python.h').is_file() or (not (include / 'pyconfig.h').is_file()):
        raise ValueError(f'Private Python headers unavailable after relocation: {include}')
    return {'python_root': str(python_root), 'include': str(include), 'ext_suffix': values['EXT_SUFFIX'], 'CC': values['CC'], 'CXX': values['CXX'], 'changed_keys': changed}

def prepare_build_toolchain(python, runtime, environment):
    """Prove that the host compiler can build and import a private Python extension."""
    env = dict(environment)
    cc, cxx = (shutil.which(name, path=env.get('PATH')) for name in ('gcc', 'g++'))
    if not cc or not cxx:
        raise RuntimeError('Native offline builds need gcc and g++; compiler preflight stopped before source installs')
    env.update(CC=cc, CXX=cxx, LDSHARED=shlex.quote(cc) + ' -pthread -shared', LDCXXSHARED=shlex.quote(cxx) + ' -pthread -shared', NVCC_CCBIN=cxx, MAX_JOBS=env.get('MAX_JOBS', '2'), PIP_DISABLE_PIP_VERSION_CHECK='1', PYTHONUNBUFFERED='1')
    configuration = repair_python_build_config(Path(python).parent.parent, env)
    for label, compiler in [('CC_version', cc), ('CXX_version', cxx)]:
        configuration[label] = subprocess.check_output([compiler, '--version'], env=env, text=True).splitlines()[0]
        print(configuration[label], flush=True)
    probe = Path(runtime) / 'build_probe'
    probe.mkdir(parents=True, exist_ok=True)
    source = probe / '_vtm_build_probe.c'
    source.write_text('#include <Python.h>\nstatic PyObject *answer(PyObject *self, PyObject *args) { return PyLong_FromLong(42); }\nstatic PyMethodDef methods[] = {{"answer", answer, METH_NOARGS, "probe"}, {NULL, NULL, 0, NULL}};\nstatic struct PyModuleDef module = {PyModuleDef_HEAD_INIT, "_vtm_build_probe", NULL, -1, methods};\nPyMODINIT_FUNC PyInit__vtm_build_probe(void) { return PyModule_Create(&module); }\n', encoding='utf-8')
    extension = probe / ('_vtm_build_probe' + configuration['ext_suffix'])
    run([cc, '-shared', '-fPIC', '-O2', '-I' + configuration['include'], source, '-o', extension], env)
    run(isolated_python_command(python, ['-c', "import sys; sys.path.insert(0,sys.argv[1]); import _vtm_build_probe; assert _vtm_build_probe.answer()==42; print('C extension build/import preflight OK')", probe]), env)
    (probe / 'toolchain.json').write_text(json.dumps(configuration, indent=2), encoding='utf-8')
    return (env, configuration)

def patch_mmcv_build_source(source):
    """Apply narrowly scoped source fixes for the pinned Torch 2.10/CUDA 13 build."""
    source = Path(source)
    changed = []
    for relative, old, new in (('setup.py', '-std=c++14', '-std=c++17'), ('mmcv/ops/csrc/pytorch/spconv_utils.h', 'tensor.type().scalarType()', 'tensor.scalar_type()')):
        path = source / relative
        if not path.is_file():
            continue
        text = path.read_text(encoding='utf-8')
        if old in text:
            path.write_text(text.replace(old, new), encoding='utf-8')
            changed.append(relative)
    return changed

def build_offline_source_wheel(bundle, record, runtime, python, environment, profile):
    """Build once with verbose logs; cache only a checksummed, ABI-specific wheel."""
    identity = {'source_sha256': record['sha256'], 'package': record['package'], 'version': record['version'], 'profile': profile, 'build_schema': 2, 'build_environment': {name: environment.get(name) for name in ('CC', 'CXX', 'LDSHARED', 'LDCXXSHARED', 'CFLAGS', 'CPPFLAGS', 'CXXFLAGS', 'LDFLAGS', 'CUDA_HOME', 'NVCC_CCBIN', 'NVCC_FLAGS', 'MMCV_WITH_OPS', 'TORCH_CUDA_ARCH_LIST')}}
    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:20]
    name = re.sub('[-_.]+', '-', record['package']).lower()
    cache = Path(runtime) / 'built_source_wheels' / name / key
    cache.mkdir(parents=True, exist_ok=True)
    receipt = cache / 'receipt.json'
    if receipt.is_file():
        try:
            saved = json.loads(receipt.read_text(encoding='utf-8'))
        except (ValueError, OSError):
            saved = {}
    else:
        saved = {}
    if 'wheel' in saved:
        wheel = checked_path(cache, saved['wheel'])
        if saved.get('identity') == identity and wheel.is_file() and (sha256(wheel) == saved.get('sha256')):
            print(f'Reuse built wheel: {wheel}', flush=True)
            return wheel
    source = prepare_build_source(bundle, record, cache / 'source_runtime')
    if name == 'mmcv-full':
        print(f'MMCV build source patches: {patch_mmcv_build_source(source)}', flush=True)
    logs = Path(runtime) / 'build_logs'
    logs.mkdir(parents=True, exist_ok=True)
    log = logs / f'{name}-{key}.pip.log'
    print(f'Build {name}; full compiler/pip log: {log}', flush=True)
    with tempfile.TemporaryDirectory(prefix='wheel-', dir=cache) as temporary:
        temporary = Path(temporary).resolve()
        if not temporary.is_relative_to(cache.resolve()):
            raise ValueError('Build staging directory escapes private cache')
        try:
            run([python, '-u', '-m', 'pip', '--log', log, '-vvv', 'wheel', '--no-index', '--no-deps', '--no-build-isolation', '--no-cache-dir', '--no-clean', '--find-links', Path(bundle) / 'wheels', '--wheel-dir', temporary, source], environment)
        except subprocess.CalledProcessError as exc:
            if name == 'polygon3':
                print('Polygon3 direct compiler diagnostic (original pip failure is retained):', flush=True)
                try:
                    run([python, '-u', 'setup.py', 'build_ext', '--verbose'], environment, cwd=source)
                except subprocess.CalledProcessError:
                    pass
            raise RuntimeError(f'Build {name} failed. Full compiler log: {log}; source kept at {source}') from exc
        wheels = list(temporary.glob('*.whl'))
        if len(wheels) != 1:
            raise RuntimeError(f'Build {name} produced {len(wheels)} wheels; inspect {log}')
        with zipfile.ZipFile(wheels[0]) as archive:
            if archive.testzip() is not None:
                raise ValueError(f'Corrupt built wheel: {wheels[0]}')
        destination = cache / wheels[0].name
        wheels[0].replace(destination)
    receipt.write_text(json.dumps({'identity': identity, 'wheel': destination.name, 'sha256': sha256(destination)}, indent=2), encoding='utf-8')
    return destination

def install_renderer(bundle, runtime):
    env = native_environment()
    common = runtime / 'common_site'
    renderer = next((bundle / 'wheels').glob('pypdfium2-5.15.0-*.whl'))
    run(host_pip_command(['install', '--no-index', '--no-deps', '--target', common, renderer]), env)
    return {'common_site': str(common)}

def install_tonggu(bundle, runtime, records):
    manifest = component_manifest(bundle, 'tonggu')
    if manifest.get('cuda') == '13.0':
        return install_tonggu_cu130(bundle, runtime, manifest)
    system = json.loads((bundle / 'system_assets_manifest.json').read_text(encoding='utf-8'))
    entries = {r['package']: r for r in system['records']}
    directory = runtime / 'tonggu_py310'
    python = directory / 'python/bin/python3.10'
    if not python.exists():
        prepare_python_runtime(bundle, entries['python-runtime'], directory)
    env = native_environment()
    bootstrap = ['pip==25.3', 'setuptools==80.10.2', 'wheel==0.48.0', 'Cython==3.0.12', 'ninja==1.13.2', 'numpy==2.2.6']
    run([python, '-m', 'pip', 'install', '--no-index', '--no-deps', '--find-links', bundle / 'wheels', *bootstrap], env)
    run([python, '-m', 'pip', 'install', '--no-index', '--no-deps', '--no-build-isolation', '--find-links', bundle / 'wheels', '--find-links', bundle / 'sources', '-r', bundle / 'requirements/tonggu.txt'], env)
    cuda = runtime / 'cuda12.8'
    for name in ('cuda_nvcc', 'cuda_cudart', 'cuda_cccl'):
        extracted = runtime / ('extract_' + name)
        extract_tar(checked_path(bundle, entries[name]['path']), extracted)
        roots = [p for p in extracted.iterdir() if p.is_dir()]
        if len(roots) != 1:
            raise ValueError(f'Unexpected CUDA archive layout: {name}')
        shutil.copytree(roots[0], cuda, dirs_exist_ok=True)
    env = native_environment(cuda)
    run([cuda / 'bin/nvcc', '--version'], env)
    if not (cuda / 'lib64').exists() and (cuda / 'lib').exists():
        (cuda / 'lib64').symlink_to('lib', target_is_directory=True)
    repo = prepare_tonggu_checkout(bundle, entries['tonggu-source'], runtime)
    built = runtime / 'built_wheels'
    built.mkdir(exist_ok=True)
    mmcv_source = checked_path(bundle, entries['mmcv-full']['path'])
    if not list(built.glob('mmcv_full-1.7.2-*.whl')):
        run([python, '-m', 'pip', 'wheel', '--no-index', '--no-deps', '--no-build-isolation', '--find-links', bundle / 'wheels', '--wheel-dir', built, mmcv_source], {**env, 'MMCV_WITH_OPS': '1', 'PIP_NO_BUILD_ISOLATION': '1'})
    mmcv = next(built.glob('mmcv_full-1.7.2-*.whl'))
    run([python, '-m', 'pip', 'install', '--no-index', '--no-deps', mmcv], env)
    run([python, repo / 'tools/patch_openmmlab.py'], env)
    probe = "import torch, mmcv, mmcv.ops, mmdet, mmrotate, json\nassert torch.version.cuda.startswith('12.8'), torch.version.cuda\nboxes = torch.tensor([[5., 5., 2., 2., 0.]], device='cuda')\nscores = torch.tensor([.9], device='cuda')\n_, keep = mmcv.ops.nms_rotated(boxes, scores, .5)\nassert keep.numel() == 1\nprint(json.dumps({'torch': torch.__version__, 'cuda': torch.version.cuda, 'mmcv': mmcv.__version__, 'sm': list(torch.cuda.get_device_capability()), 'rotated_nms_gpu': 'OK'}))\n"
    run([python, '-c', probe], env)
    return {'tonggu_python': str(python), 'tonggu_repo': str(repo), 'tonggu_compiled_ops_verified': True, 'tonggu_cuda_home': str(cuda)}

def install_tonggu_cu130(bundle, runtime, manifest):
    system = component_manifest(bundle, 'system_assets')
    entries = {record['package']: record for record in system['records']}
    directory = runtime / 'tonggu_py310_cu130'
    python = directory / 'python/bin/python3.10'
    if not python.exists():
        prepare_python_runtime(bundle, entries['python-runtime'], directory)
    nvcc = shutil.which('nvcc')
    if not nvcc:
        raise RuntimeError('TongGu needs the CUDA 13.0 toolkit verified in the Kaggle report')
    candidates = [Path(value) for value in (os.environ.get('CUDA_HOME'), os.environ.get('CUDA_PATH')) if value]
    candidates += [Path(nvcc).resolve().parents[1], Path('/usr/local/cuda'), Path('/usr/local/cuda-13.0')]
    cuda = next((path.resolve() for path in candidates if (path / 'include/cuda.h').is_file()), candidates[0].resolve())
    compiler = subprocess.check_output([nvcc, '--version'], text=True)
    if 'release 13.0' not in compiler or not (cuda / 'include/cuda.h').is_file():
        raise RuntimeError(f'Expected a complete CUDA 13.0 toolkit, found {cuda}')
    env = native_environment(cuda)
    site, version = target_site(python, runtime)
    env, toolchain = prepare_build_toolchain(python, runtime, env)
    clear_reused_packages(site, private_kernel_cleanup_records(manifest))
    initial_runner = private_python(python, site)
    run(host_pip_command(['--python', initial_runner, 'install', '--no-index', '--no-deps', '--find-links', bundle / 'wheels', '-r', bundle / 'requirements/tonggu.txt']), env)
    reused = reuse_kernel_packages(manifest.get('reused', []), site, version, bundle=bundle, manifest=manifest)
    runner = private_python(python, site)
    ensure_private_opencv(bundle, manifest, site, runner, env, runtime)
    torch_probe = "import torch; assert torch.version.cuda == '13.0', torch.version.cuda; x=torch.ones((2,2),device='cuda'); assert float((x@x).sum().item()) == 8.0; print(torch.__version__,torch.cuda.get_device_capability())"
    run([runner, '-c', torch_probe], env)
    configure_cuda_architecture(runner, env)
    versions = {record['package']: record['version'] for record in [*manifest['records'], *reused]}
    profile = {'python': version, 'numpy': versions.get('numpy'), 'torch': versions.get('torch'), 'cuda': '13.0', 'arch': env['TORCH_CUDA_ARCH_LIST'], 'toolchain': {k: v for k, v in toolchain.items() if k != 'changed_keys'}}
    for record in manifest['records']:
        if record['kind'] != 'wheel' and record['package'] != 'mmcv-full':
            wheel = build_offline_source_wheel(bundle, record, runtime, runner, env, profile)
            run([runner, '-m', 'pip', 'install', '--no-index', '--no-deps', '--force-reinstall', wheel], env)
    source_probe = "import Polygon, numpy as np, jieba, zhconv, cv2\nassert Polygon.withNumPy, 'Polygon must be built with the selected NumPy headers'\np = Polygon.Polygon(np.array([[0.,0.],[2.,0.],[2.,2.],[0.,2.]], dtype=np.float64))\nassert abs(p.area()-4.) < 1e-9\nassert cv2.resize(np.zeros((2,2,3), dtype=np.uint8),(4,4)).shape == (4,4,3)\nprint('Polygon/NumPy, jieba, zhconv, OpenCV native imports/operations OK')\n"
    run([runner, '-c', source_probe], env)
    repo = prepare_tonggu_checkout(bundle, entries['tonggu-source'], runtime)
    mmcv_source = next((record for record in manifest['records'] if record['package'] == 'mmcv-full'))
    mmcv = build_offline_source_wheel(bundle, mmcv_source, runtime, runner, {**env, 'MMCV_WITH_OPS': '1', 'PIP_NO_BUILD_ISOLATION': '1'}, profile)
    run([runner, '-m', 'pip', 'install', '--no-index', '--no-deps', '--force-reinstall', mmcv], env)
    run([runner, repo / 'tools/patch_openmmlab.py'], env)
    run([runner, '-m', 'pip', 'check'], env)
    probe = "import torch, mmcv, mmcv.ops, mmdet, mmrotate, json\nb = torch.tensor([[5.,5.,2.,2.,0.]], device='cuda')\ns = torch.tensor([.9], device='cuda')\n_, keep = mmcv.ops.nms_rotated(b,s,.5)\nassert keep.numel() == 1\nprint(json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,'mmcv':mmcv.__version__,'rotated_nms_gpu':'OK'}))\n"
    run([runner, '-c', probe], env)
    return {'tonggu_python': str(runner), 'tonggu_repo': str(repo), 'tonggu_cuda_home': str(cuda), 'tonggu_compiled_ops_verified': True, 'tonggu_kernel_reused': reused}



_INSTALL_OUTPUT = None

def run(args, environment=None, *, cwd=None):
    """Chạy lệnh cài đặt; log đầy đủ ghi ra file do caller mở."""
    print("RUN:", command_description(args), flush=True)
    subprocess.run(list(map(str, args)), env=environment, cwd=cwd, check=True,
                   stdout=_INSTALL_OUTPUT, stderr=subprocess.STDOUT if _INSTALL_OUTPUT else None)

# ==== Chuẩn bị runtime TongGu + package vtm_corpus (dùng lại nếu khỏe) ====
def _runtime_state(runtime_dir):
    config = Path(runtime_dir) / 'runtime_config.json'
    return json.loads(config.read_text(encoding='utf-8')) if config.is_file() else {}

def _tonggu_runtime_ready(runtime_dir, log_dir):
    state = _runtime_state(runtime_dir)
    py, repo = (state.get('tonggu_python'), state.get('tonggu_repo'))
    if not (py and Path(py).is_file() and repo and (Path(repo) / 'tongguocr/pipeline.py').is_file()):
        return None
    check = "import torch, mmcv.ops, mmdet, mmrotate\nb = torch.tensor([[5.,5.,2.,2.,0.]],device='cuda')\ns = torch.tensor([.9],device='cuda')\nassert mmcv.ops.nms_rotated(b, s, .5)[1].numel() == 1\nprint('tonggu runtime OK')"
    try:
        result = subprocess.run([py, '-c', check], capture_output=True, text=True, encoding='utf-8',
                                errors='replace', timeout=120, env={**os.environ, 'MPLBACKEND': 'Agg'})
        (Path(log_dir) / 'cache_tonggu.log').write_text((result.stdout or '') + '\n' + (result.stderr or ''), encoding='utf-8')
        return state if result.returncode == 0 else None
    except (OSError, subprocess.SubprocessError):
        return None

def prepare_tonggu_runtime(*, packages_dir='', runtime_dir='/tmp/vtm_tonggu_runtime', log_dir='/kaggle/working/offline_install_logs', force_reinstall=False):
    log_dir = Path(log_dir); log_dir.mkdir(parents=True, exist_ok=True)
    runtime = Path(runtime_dir); runtime.mkdir(parents=True, exist_ok=True)
    ready = None if force_reinstall else _tonggu_runtime_ready(runtime, log_dir)
    if ready:
        print('Runtime TongGu khỏe, dùng lại:', ready['tonggu_python'])
    else:
        bundle = find_packages_dir(packages_dir)
        print('Bundle packages:', bundle)
        records = verify(bundle, report_path=log_dir / 'source_tree_report.json')
        state = install_tonggu(bundle, runtime, records)
        support = install_renderer(bundle, runtime)
        state = {**state, 'common_site': support['common_site']}
        (runtime / 'runtime_config.json').write_text(json.dumps(state, indent=2) + '\n', encoding='utf-8')
        ready = state
        print('TongGu runtime OK:', state['tonggu_python'])
    # Cài wheel mã dự án vào cả Python riêng của TongGu (worker chạy `python -m vtm_corpus.ocr_worker`).
    wheel = _find_vtm_wheel()
    run(host_pip_command(['--python', ready['tonggu_python'], 'install', '--no-index', '--no-deps', '--force-reinstall', wheel]), native_environment())
    ready['vtm_corpus_wheel'] = str(wheel)
    print('vtm_corpus wheel (private Python):', wheel.name)
    # Worker render PDF bằng pypdfium2; PYTHONPATH bị -I bỏ qua nên cài thẳng vào private site.
    pdfium = _walk_input(lambda f: f.startswith('pypdfium2-') and f.endswith('.whl'), max_hits=1)
    if pdfium:
        run(host_pip_command(['--python', ready['tonggu_python'], 'install', '--no-index', '--no-deps', '--force-reinstall', pdfium[0]]), native_environment())
        print('pypdfium2 (private Python):', pdfium[0].name)
    return ready

def find_packages_dir(packages_dir=''):
    if packages_dir:
        bundle = Path(packages_dir)
        if (bundle / 'paddle_manifest.json').is_file() and (bundle / 'system_assets_manifest.json').is_file():
            return bundle.resolve()
        if (bundle / 'packages/paddle_manifest.json').is_file():
            return (bundle / 'packages').resolve()
        raise FileNotFoundError(f'PACKAGES_DIR chưa đúng: thiếu manifest trong {bundle}')
    matches = [parent for parent, dirs, files in _walk_dirs()
               if {'paddle_manifest.json', 'system_assets_manifest.json'} <= set(files)]
    if len(matches) != 1:
        raise RuntimeError(f'Tìm thấy {len(matches)} Dataset packages. Điền PACKAGES_DIR ở ô cấu hình.')
    return matches[0].resolve()


### Xem danh sách PDF đầu vào

Chạy trước khi cấu hình: quét `/kaggle/input` để tìm thư mục chứa PDF và liệt kê
sách/từng trang. Dùng kết quả này để điền `INPUT_DIR` và `books` ở bước 2.


In [ ]:
# ==== XEM DANH SÁCH PDF ĐẦU VÀO (trước cấu hình) ====
# Tự quét /kaggle/input tìm thư mục chứa PDF (bỏ qua Dataset packages/models),
# in danh sách sách để điền `DATA_DIR` và `BOOKS` ở ô cấu hình bên dưới.
#   books=[]                          -> tất cả sách
#   books=["Vệ_sinh_yếu_chỉ_-_tập_4"] -> đúng 1 sách
#   books=["*ky_huyet*", "*nhi*"]     -> nhiều sách theo glob
def _candidate_pdf_dirs(root='/kaggle/input'):
    dirs = []
    for parent, dirs_, files in _walk_dirs(root):
        pdfs = [f for f in files if f.lower().endswith('.pdf')]
        if pdfs and 'paddle_manifest.json' not in files:
            dirs.append(Path(parent))
            dirs_[:] = []
    return dirs

_candidates = _candidate_pdf_dirs()
print('Thư mục chứa PDF tìm thấy:')
for d in _candidates:
    print(' -', d)
if not _candidates:
    print('Không thấy PDF nào trong /kaggle/input; kiểm tra Dataset PDF đã gắn.')
for _dir in _candidates:
    try:
        _entries = list_pdfs(_dir)
    except FileNotFoundError:
        continue
    print(f'\n{_dir} ({len(_entries)} sách):')
    for i, e in enumerate(_entries, 1):
        print(f"{i:3}. {e['name']:<50} {e['n_pages']:>4} trang  {e['path']}")


## 2. Cấu hình (một ô duy nhất)

Sửa mọi tham số tại đây. Ô này cũng cài runtime TongGu offline (`prepare_tonggu_runtime`)
và ghi `tonggu_paths.json` trỏ tới weights trong Dataset models. Chạy thử nhanh:
`pages = "1-2"`, `limit = 2`; chạy đầy đủ: `pages = ""`, `limit = None`.


In [ ]:
# ==== CẤU HÌNH OCR — sửa mọi trường ngay tại đây ====
from vtm_corpus import OCRConfig, ReportConfig

INPUT_DIR = '/kaggle/input/datasets/nthday/vietnamese-traditional-medicine'   # thư mục chứa PDF (xem danh sách ở ô trên)
PACKAGES_DIR = ''   # để rỗng = tự tìm Dataset chứa manifest packages
MODELS_DIR = ''     # để rỗng = tự tìm Dataset chứa weights TongGu (tonggu/ocr + tonggu/layout)
WORK_DIR = '/kaggle/working'

# Cài runtime TongGu offline từ Dataset packages (private Python 3.10 + CUDA + mmcv),
# cài lại wheel vtm_corpus vào Python riêng của worker; runtime khỏe sẽ được dùng lại.
TONGGU = prepare_tonggu_runtime(packages_dir=PACKAGES_DIR)

# Tự tìm weights TongGu trong Dataset models: tonggu/ocr (model OCR) và
# tonggu/layout/finetune_infer.pth (detector layout).
def _find_tonggu_weights():
    if MODELS_DIR and (Path(MODELS_DIR) / 'tonggu/ocr/config.json').is_file():
        return Path(MODELS_DIR)
    hits = []
    for parent, dirs, files in _walk_dirs():
        if 'config.json' in files:
            path = Path(parent)
            if path.parts[-2:] == ('tonggu', 'ocr'):
                hits.append(path)
                dirs[:] = []
    if not hits:
        raise FileNotFoundError('Không thấy tonggu/ocr/config.json trong /kaggle/input; kiểm tra Dataset models.')
    root = hits[0].parents[1]
    assert (root / 'tonggu/layout/finetune_infer.pth').is_file(), f'Thiếu {root}/tonggu/layout/finetune_infer.pth'
    return root

_models_root = _find_tonggu_weights()
print('Weights TongGu:', _models_root)
Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
TONGGU_CONFIG = Path(WORK_DIR) / 'tonggu_paths.json'
TONGGU_CONFIG.write_text(json.dumps({
    'layout_inference': {'detector_checkpoint_path': str(_models_root / 'tonggu/layout/finetune_infer.pth')},
    'ocr': {'transformers': {'model_path': str(_models_root / 'tonggu/ocr')}},
}, indent=2) + '\n', encoding='utf-8')

OCR = OCRConfig(
    engine="tonggu",                  # engine duy nhất của notebook này
    input_dir=INPUT_DIR,              # thư mục chứa PDF (glob đệ quy)
    books=[],                         # [] = tất cả; tên/glob, vd ["*ky_huyet*"]
    workdir=WORK_DIR,                 # nơi tạo runs/
    run_name="tonggu_stage3",         # "" = dùng engine
    pages="",                         # "" = cả sách; "1-10,15" = chạy thử
    dpi=200,                          # tăng nếu sách chữ nhỏ, vd 225
    max_side=1280,                    # sách chữ dọc nhỏ nên giảm, vd 750
    segmentation="auto",              # auto: TongGu giữ layout gốc, tách cột khi vượt ngân sách
    reading_direction="vertical_rtl", # vertical_rtl | horizontal_ltr (sách ngang)
    model_min_pixels=1024,
    model_max_pixels=1048576,
    max_new_tokens=6000,
    dtype="fp16",
    model="TongGuOCR-Stage3",
    prompt="",                        # "" = prompt mặc định
    repetition_penalty=1.08,
    device="auto",                    # auto | cpu | cuda | cuda:1
    limit=None,                       # vd 2 = chỉ chạy 2 trang để thử
    page_timeout=1800,                # TongGu nạp model trước timeout; 1800 giây/trang
    n_gpu=0,                          # 0 = tự phát hiện; 4x T4 sẽ sharding trang qua 4 worker
    worker_python=TONGGU['tonggu_python'],   # worker chạy trong private Python 3.10 của TongGu
    budget_h=10.5,
    stall_min=15,
    max_restarts=5,
    n_layout=0,                       # layout vẽ cho hunyuan/paddle; tonggu để 0
    export=True,
    fresh=False,                      # True = xoá run cũ cùng cấu hình
    resume_from=[],                   # vd ["/kaggle/input/<notebook-output>"]
)

REPORT = ReportConfig(
    preview_pages=3,
    preview_chars=1500,
    layout_images=6,
    show_tables=True,
    image_width=700,
)


## 3. Chạy OCR

Checkpoint được lưu sau mỗi trang; chạy lại cùng cấu hình dùng kết quả đã có; trang
lỗi được thử lại trong lượt chạy mới. TongGu nạp model trước timeout trang (mặc định
1.800 giây/trang). Crop lấy từ ảnh render gốc; `segmentation: auto` giữ layout gốc
của TongGu.


In [ ]:
# ==== CHẠY OCR ====
run = run_ocr(OCR)
print("run_dir:   ", run.run_dir)
print("output_dir:", run.output_dir)
run


## 4. Xem kết quả ngay trong notebook

Báo cáo theo run, xem text từng trang. Output chính: `runs/ocr_*/txt/<tên sách>.txt`
và `ocr_raw/` (checkpoint JSONL).


In [ ]:
# ==== XEM KẾT QUẢ NGAY TRONG NOTEBOOK ====
show_ocr_report(run.run_dir, REPORT)
show_pages(run.run_dir, REPORT)


Ghi chú chạy thử nhanh: dùng ô ở cuối. Đồng thuận/so sánh giữa các model nằm ở
notebook `3_` (nhánh `mduy`); notebook này chỉ chạy TongGu một model.


In [ ]:
# ==== CHẠY THỬ NHANH (tuỳ chọn) ====
# Bỏ comment để chạy 2 trang đầu rồi xem kết quả.
# OCR.pages = "1-2"
# OCR.limit = 2
# run = run_ocr(OCR)
# show_ocr_report(run.run_dir, REPORT)
# show_pages(run.run_dir, REPORT)
